# Notebook 05 — RAG retrieval inspection

Tests the retrieval pipeline in isolation — **no language model needed**.
Useful for debugging before training anything.

**What you can do here:**
- Check that your chunks.json was built correctly
- See what the retriever actually returns for a query
- Tune `top_k` and see how results change
- Filter by chapter/topic
- Spot gaps — topics where retrieval returns weak results (low scores)

**Run this FIRST** before debugging any model — most "bad answers" are
actually bad retrieval.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════

RAG_INDEX_DIR = "../teaching_assistant/rag_index"
CHUNKS_JSON   = "../teaching_assistant/rag_index/chunks.json"

# Query to test
QUERY = "How does dropout prevent overfitting?"

# How many results to fetch
TOP_K = 5

# Optional filters (set to None to disable)
FILTER_H1 = None   # e.g. "Neural Networks" — only return chunks from this chapter
FILTER_H2 = None   # e.g. "Dropout"
FILTER_SOURCE = None   # e.g. "lecture_03" — filter by filename substring

In [ ]:
# ═══════════════════════════════════════════════════════════
# LOAD RETRIEVER
# ═══════════════════════════════════════════════════════════
import sys
sys.path.insert(0, "../teaching_assistant")
from retriever import Retriever

retriever = Retriever(index_dir=RAG_INDEX_DIR)
print("Retriever ready")

In [ ]:
# ═══════════════════════════════════════════════════════════
# RUN A QUERY
# ═══════════════════════════════════════════════════════════

results = retriever.query(
    QUERY,
    top_k=TOP_K,
    filter_h1=FILTER_H1,
    filter_h2=FILTER_H2,
    filter_source=FILTER_SOURCE,
)

print(f'Query: "{QUERY}"')
print(f"Got {len(results)} results\n")

for i, r in enumerate(results, 1):
    print(f"{'─'*55}")
    print(f"#{i}  score={r['score']:.3f}  |  {r.get('breadcrumb', '(no heading)')}")
    print(f"    source: {r['source']}")
    # Show first 250 chars of the chunk
    preview = r['text'][:250].replace("\n", " ")
    print(f"\n    {preview}...\n")

In [ ]:
# ═══════════════════════════════════════════════════════════
# EXPLORE THE CURRICULUM MAP
# See what chapters and topics are in your index.
# ═══════════════════════════════════════════════════════════
import json
from collections import defaultdict

with open(CHUNKS_JSON) as f:
    chunks = json.load(f)

print(f"Total chunks in index: {len(chunks)}\n")

# Build chapter → topic → count map
curriculum = defaultdict(lambda: defaultdict(int))
for c in chunks:
    h1 = c.get("metadata", {}).get("h1") or "(no h1)"
    h2 = c.get("metadata", {}).get("h2") or "(no h2)"
    curriculum[h1][h2] += 1

for h1, topics in sorted(curriculum.items()):
    total = sum(topics.values())
    print(f"  {h1}  ({total} chunks)")
    for h2, n in sorted(topics.items()):
        print(f"      └─ {h2}: {n} chunk(s)")

In [ ]:
# ═══════════════════════════════════════════════════════════
# BATCH QUERY: test several topics at once
# Useful to spot topics where retrieval is weak (score < 0.4
# means the chunks aren't very relevant — possibly a gap in
# your course material or a chunking issue).
# ═══════════════════════════════════════════════════════════

test_queries = [
    "What is dropout?",
    "How does backpropagation work?",
    "Explain the attention mechanism.",
    "What is a convolutional neural network?",
    "Gradient descent optimization",
]

print(f"{'Query':<45} {'Top score':>10} {'Breadcrumb'}")
print("─" * 80)

for q in test_queries:
    res = retriever.query(q, top_k=1)
    if res:
        top = res[0]
        print(f"{q:<45} {top['score']:>10.3f}  {top.get('breadcrumb','?')[:30]}")
    else:
        print(f"{q:<45} {'no results':>10}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# SHOW A FULL CHUNK
# Inspect any chunk by its index to see the full text.
# ═══════════════════════════════════════════════════════════

CHUNK_INDEX = 0   # change to inspect a different chunk

c = chunks[CHUNK_INDEX]
print(f"Chunk {CHUNK_INDEX}")
print(f"Source:    {c.get('source', '?')}")
print(f"Metadata:  {c.get('metadata', {})}")
print(f"\nText ({len(c['text'])} chars):")
print(c['text'])